In [1]:
# goal: prototype skeleton transformer
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from map4 import MAP4
import re

import matplotlib.pyplot as plt
%matplotlib inline

from pathlib import Path

In [2]:
import numpy as np
import torch

class RadialSeeker:

    def __init__(self, radial_resolution, intrashell_resolution, max_angstroms,
                 verbose=False):
        self.radial_resolution = radial_resolution
        self.angstrom_lim = max_angstroms   # maximum angstrom range found + some extra for context enrichment
                                 # can always edit this later on
        self.angstrom_inc = float(max_angstroms / radial_resolution)
        self.threshold = self.angstrom_inc / 2 # standardize how we determine radial sequences
        # getting half the angstrom increment will let us finely separate them

        #                           smallest distance is the base increment, not 0
        self.radius_levels = torch.arange(self.angstrom_lim, self.angstrom_inc, step= -1 * self.angstrom_inc)

        self.intrashell_resolution = intrashell_resolution
        self.intrashell_inc = max_angstroms / intrashell_resolution

        self.verbose = verbose

    def radial_sequence(self, aa_seq, vect_seq):
        # first order them by absolute distance to centroid
        # then convert them to indices according to intra-shell resolution

        # radial resolution determines how finely we want to order them
        # shell resolution determines how finely in 3D space we represent them

        radial_seq = []  # lookup table
        seen = set()

        # sanity
        if len(aa_seq) != len(vect_seq):
            raise ValueError(f"string and vector sequences of {aa_seq} are different lengths")

        # iterate up resolution increments, if a molecule's coordinate is within like 1/resolution of an angstrom, append it's info
        # its (radius, pos index) is now the unique ID, so if we stumble on it again its not duplicated
        for level in self.radius_levels: # go through all possible radius levels
            for i in range(len(aa_seq)):  # go through all amino acids in that sequence
                num_id=i
                if num_id not in seen and self._dist_check(np.linalg.norm(vect_seq[i]), level):
                    idx_vect = self.vect2idx(vect_seq[i])
                    radial_seq.append([[aa_seq[i]], idx_vect])
                    seen.add(num_id)

        # create two VOID tokens (0,0,0) on both sides to denote stops and starts
        return radial_seq + [[['VOID'],[0, 0, 0]]]  # we want to go from outside inward

    def _dist_check(self, dist, ang_radius):
        if abs(dist - ang_radius) <= self.threshold:
            return True
        else:
            return False

    def vect2idx(self, vector):
        """
        ENCODE: Converts a torch.tensor of 3D vectors into their nearest shell-resolution's index
        """
        idxs = np.round(vector / self.intrashell_inc)
        idxs = np.clip(idxs, -self.intrashell_resolution, self.intrashell_resolution)
        return idxs.astype(int)

    def num2vect(self, idxs):
        """DECODE: Converts a torch.tensor of indexes into their max_Angstrom-scaled 3D vectors"""
        vector = []
        for number in idxs:
            vector.append(float(number) * self.angstrom_inc)
        return vector



def test_resolution():
    module = RadialSeeker(radial_resolution = 100, intrashell_resolution = 100, max_angstroms = 33)
    for level in module.radius_levels:
        print(level)
# test_resolution()  # all good

In [3]:
mol_dim = 1024
map4_fprinter = MAP4(
    dimensions=mol_dim,
    radius=2,
    include_duplicated_shingles=True,
)

# will be important later for visualization of skeleton
def eval_lipinski(smiles):
    """
    Use on user-input smiles, don't shut down inference run just flag molecule as non drug-like
    :param smiles:
    :return:
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES String")

    try:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Lipinski.NumHDonors(mol)
        h_acceptors = Lipinski.NumHAcceptors(mol)
    except Exception as e:
        raise ValueError(f"Error calculating Lipinski Descriptors: {e}")

    rules_passed = 0
    if mw <= 500: rules_passed += 1
    if logp <= 5: rules_passed += 1
    if h_donors <= 5: rules_passed += 1
    if h_acceptors <= 10: rules_passed += 1

    return rules_passed

def mol_from_smiles(smiles):
    """
    Extract molecular fingerprint from singular SMILES - fairly straightforward
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        map4_fp = map4_fprinter.calculate(mol=mol)

        return map4_fp

    except Exception as e:
        return None

In [10]:

# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
data_folder = Path("notebook_database")
RADIAL_SEQUENCES = pd.read_pickle(data_folder/"radial_seq_df.pkl")
MOLECULAR_FINGERPRINTS = pd.read_pickle(data_folder/"molfp_df.pkl")
SMILE_2_PDBHITS = pd.read_csv(data_folder/"SMILE_2_PDBhits.csv")

Index(['PDB_ID', 'radial_sequence'], dtype='str')


In [5]:
# check entries
print(SMILE_2_PDBHITS["PDB_hits"].head(1).to_string())
print(type(MOLECULAR_FINGERPRINTS["map4_fp"].iloc[0]))
print(type(RADIAL_SEQUENCES["radial_sequence"].iloc[0]))

0    ['9DHL', '9C84', '9C81', '9C6P', '6YPD', '6YEO...
<class 'torch.Tensor'>
<class 'list'>


In [6]:
# RadialSeeker parameters used -> record these - might design a config file later
radial_resolution = 100
intrashell_resolution = 100
max_angstroms = 33

torch.manual_seed(311104)

In [7]:
# need to think of how we're about to load up our data
# i guess a nested for loop is how we get every combination the easiest?
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
test_radial_seq = RADIAL_SEQUENCES.iloc[0]["radial_sequence"]

def radial_XY(radial_seq, block_size):
    """
    Generates X and Y set for a given radial sequence
    :param radial_seq:
    :param block_size:
    :return:
    """
    X, Y = [], []
    context = [[0,0,0]] * block_size
    for idx in radial_seq:
        coords = idx[1]   # = [X, Y, Z]
        X.append(context)
        Y.append(coords)
        context = context[1:] + [coords]

    return X, Y

X, Y = radial_XY(radial_seq=test_radial_seq, block_size=10)
for pos in range(len(X)):
    print(f"{X[pos]} => {Y[pos]}")

[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]] => [-32, -18, 22]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22]] => [-24, 31, 5]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5]] => [23, -7, 28]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28]] => [31, -8, 12]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28], [31, -8, 12]] => [-4, 28, -16]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28], [31, -8, 12], [-4, 28, -16]] => [-3, 19, 25]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28], [31, -8, 12], [-4, 28, -16], [-3, 19, 25]] => [-26, 16, -8]
[[0, 0, 0], [0, 0, 0], [0, 0, 

In [8]:
# noteblock:
# Generate X and Y total
# There will be two Xs, an X_rad and an X_mfp
# X_rad: X data for radial sequence
# X_mfp: X data for molecular fingerprint
# Query source = SMILES fingerprint
# Key / Value Source = Protein Coordinates

In [18]:
import ast
# X and Y data generator -> for now let's just use one PDB and SMILES
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
testset = SMILE_2_PDBHITS.iloc[0]
# query for smiles
sample_smiles = testset["SMILES"]
sample_hit = MOLECULAR_FINGERPRINTS.loc[MOLECULAR_FINGERPRINTS["smiles_str"]==sample_smiles]
# query for pdb radial sequence
sample_pdb = ast.literal_eval(testset["PDB_hits"])[0]
sample_radial_entry = RADIAL_SEQUENCES.loc[RADIAL_SEQUENCES["PDB_ID"]==sample_pdb]


sample_molfp = sample_hit["map4_fp"].iloc[0]  # for some reason not using iloc returns a series not a string....
sample_radial_sequence = sample_radial_entry["radial_sequence"]
print(len(sample_radial_sequence))

print(sample_molfp)
print(sample_radial_sequence)


1
tensor([0., 1., 0.,  ..., 1., 0., 1.])
11977    [[[E], [39, -5, 33]], [[R], [-5, -16, 44]], [[...
Name: radial_sequence, dtype: object


In [19]:
print(RADIAL_SEQUENCES.columns)

Index(['PDB_ID', 'radial_sequence'], dtype='str')


In [20]:
sample_radialX, sampleY = radial_XY(sample_radial_sequence, block_size=10)
for pos in range(len(X)):
    print(f"{X[pos]} => {Y[pos]}")

[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]] => [-32, -18, 22]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22]] => [-24, 31, 5]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5]] => [23, -7, 28]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28]] => [31, -8, 12]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28], [31, -8, 12]] => [-4, 28, -16]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28], [31, -8, 12], [-4, 28, -16]] => [-3, 19, 25]
[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-32, -18, 22], [-24, 31, 5], [23, -7, 28], [31, -8, 12], [-4, 28, -16], [-3, 19, 25]] => [-26, 16, -8]
[[0, 0, 0], [0, 0, 0], [0, 0, 

In [71]:
batch_size = 32
block_size = 1
max_iters = 500
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 3
dropout = 0.2
coord_num = 101

In [13]:
# note
# key value tensor -> molecular fingerprint
# query tensor -> protein coordinate context

In [14]:
sample_radialX = torch.tensor(X)
sample_radialX = torch.tensor(sample_radialX, dtype=torch.float32)
sampleY = torch.tensor(sampleY)
sample_molfp  # this is already a tensor so we're good on that for context

/tmp/ipykernel_215925/2553057511.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sample_radialX = torch.tensor(sample_radialX, dtype=torch.float32)


tensor([0., 0., 1.,  ..., 0., 0., 0.])

In [62]:
print(sample_radialX.shape)

torch.Size([32, 10, 3])


In [72]:
# data loading
class Head(nn.Module):
    """one head of self-attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)     # linear projections
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # this is just what you have to do to create the lower triangular matrix thing
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)   # both (B, T, C)

        # compute attention scores ("affinities")
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, C) @ (B, C, T) -> (B, T, T)
        attn_wts = attn_wts.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighted aggregations
        v = self.value(x)
        out = attn_wts @ v  # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


class MultiHeadAttention(nn.Module):  # later will change this to MH-cross-attention
    """multiple heads of self-attention in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        # introduce a projection
        # run all of these single head self attention heads in parallel into a list
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)  # output of self.attention itself + applying a linear transformation of the outcome
        # concatenate all of the outputs
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),   # projection layer going back into the residual pathway
            nn.Dropout(dropout)
        )
        # multiplied n_embd by 4 because that's what the paper did
        # adding more computation and growing the layer in the residual block on the side

    def forward(self, x):
        return self.net(x)


class LayerNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        # backprop-trained parameters
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        # calculate forward pass
        xmean = x.mean(0, keepdim=True)
        xvar = x.var(0, keepdim=True)
        xhat = (x-xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.satt_head = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # applied layer norm pre-formulation
        x = x + self.satt_head(self.ln1(x))  # residual connections
        x = x + self.ffwd(self.ln2(x))
        # for these two, we're basically forking off, doing some computation and returning
        return x

In [117]:
# structure

class Skeleton(nn.Module):
    def __init__(self, n_context):
        super().__init__()
        self.coordinates_in = nn.Linear(3, n_embd)
        self.position_embedding = nn.Embedding(n_context, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)

        self.coordinates_out = nn.Linear(n_embd, 3)


    def forward(self, idx, targets=None):
        B, T, _ = idx.shape  # B, T, C -> triplets
        coord_emb = self.coordinates_in(idx)
        position_emb = self.position_embedding(torch.arange(T))
        x = coord_emb + position_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.coordinates_out(x)

        if targets is None:
            loss = None
        else:
            last_pred = logits[:, -1, :]
            loss = F.mse_loss(last_pred, targets.float())

        return logits, loss


    def generate(self, coord_context):
        coordinates = []
        molecular_context = None   # will set molecular fingerprint in here
        while True:
            idx_cond = coord_context
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            coordinate_sample = torch.multinomial(probs, num_samples=1)
            print(f"Coordinate_sample: {coordinate_sample}")
            coordinates.append(coordinate_sample)
            if 0 in coordinate_sample:
                break
        return coordinates


model = Skeleton(10)
print(sample_radialX)
print(sampleY)


tensor([[[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [35., 38., 37.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
      

In [75]:
n = int(0.9 * len(sample_radialX))
train_data = sample_radialX[:n]
val_data = sample_radialX[n:]

def get_batch(split):
    """generate small batch of data inputs x and targets y"""
    data = train_data if split == 'train' else val_data  # gets us a data array
    ix = torch.randint(len(data) - block_size, (batch_size,))  # generate random positions to sample
    x = sample_radialX[ix]
    y = sampleY[ix]
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



In [76]:
print(type(train_data))
print(type(val_data))

print(train_data.shape)
print(val_data.shape)

<class 'torch.Tensor'>
<class 'torch.Tensor'>
torch.Size([28, 10, 3])
torch.Size([4, 10, 3])


In [80]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    losses = estimate_loss()
    print(losses)

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

{'train': tensor(2431.4019), 'val': tensor(1664.9800)}
{'train': tensor(2442.1187), 'val': tensor(1639.0715)}
{'train': tensor(2435.4146), 'val': tensor(1627.1072)}
{'train': tensor(2431.3103), 'val': tensor(1641.2085)}
{'train': tensor(2411.9553), 'val': tensor(1643.4552)}
{'train': tensor(2408.7119), 'val': tensor(1664.1488)}
{'train': tensor(2401.8318), 'val': tensor(1656.0995)}
{'train': tensor(2402.0681), 'val': tensor(1628.0847)}
{'train': tensor(2387.0696), 'val': tensor(1635.2306)}
{'train': tensor(2378.4941), 'val': tensor(1615.6228)}
{'train': tensor(2370.2046), 'val': tensor(1638.0905)}
{'train': tensor(2351.2788), 'val': tensor(1611.3285)}
{'train': tensor(2349.0979), 'val': tensor(1627.5620)}
{'train': tensor(2352.9685), 'val': tensor(1624.7198)}
{'train': tensor(2331.4565), 'val': tensor(1596.4940)}
{'train': tensor(2339.9341), 'val': tensor(1601.2217)}
{'train': tensor(2325.9912), 'val': tensor(1611.9751)}
{'train': tensor(2326.1492), 'val': tensor(1609.4192)}
{'train': 

AttributeError: 'RadialSeeker' object has no attribute 'num2vect'

In [158]:
DecoderEncoder = RadialSeeker(100, 100, 33)
context = torch.zeros(1, 10, 3)
print(DecoderEncoder.num2vect(model.generate(context)))

Coordinate_sample: tensor([[2]])
Coordinate_sample: tensor([[0]])
[0.66, 0.0]


In [ ]:
class CrossAttentionHead(nn.Module):

    def __init__(self, n_embd, head_size, dropout=dropout):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x, context):
        query = self.query(x)
        key = self.key(context)
        value = self.value(context)

        head_size = query.shape[-1]

        attention = query @ key.transpose(-2, -1) * head_size **-0.5
        attention = F.softmax(attention, dim=-1)
        attention = self.dropout(attention)

        return attention @ value

In [21]:
print(type(model))

<class '__main__.Skeleton'>
